In [20]:
import jax.numpy as jnp
import jax
from jax import grad, jacfwd
import numpy as np

In [41]:
def W_NH_1d(F):
    # F: deformation gradient (scalar)
    return 0.5*(F**2 - 1.0) + 1.5*(F - 1)**2

In [42]:
def make_bar(L, n_ele) :
    coords = jnp.linspace(0, L, n_ele + 1)
    node_idx = jnp.arange(0, coords.shape[0])
    cells = jnp.stack([node_idx[:-1], node_idx[1:]]).T
    return coords, cells
def linear_basis(xi):
    # xi in [-1,1] -> shape functions and derivatives wrt xi
    N = jnp.array([0.5*(1 - xi), 0.5*(1 + xi)])
    dN_dxi = jnp.array([-0.5, 0.5])
    return N, dN_dxi

In [43]:
GAUSS_POINTS = jnp.array([-1/jnp.sqrt(3.), 1/jnp.sqrt(3.)])
GAUSS_WEIGHTS = jnp.array([1.0, 1.0])

In [44]:
def element_energy(u_e, X_e, traction_local=0.0):
    # u_e: 2 nodal displacements for the element (array shape (2,))
    # X_e: 2 node positions in reference config
    # traction_local: traction work contribution (for Neumann on right end)
    # compute Pi_e = ∫ W(F) dX - traction * u(right) (traction contribution handled externally)
    # We'll integrate using Gauss points
    Pi = 0.0
    # element length in reference coords: Jacobian mapping dX/dxi = dx_dxi
    dx_dxi = (X_e[1] - X_e[0]) / 2.0
    for xi, w in zip(GAUSS_POINTS, GAUSS_WEIGHTS):
        N, dN_dxi = linear_basis(xi)
        # physical derivative: dN/dX = dN/dxi * 1/(dX/dxi)
        dN_dX = dN_dxi / dx_dxi
        # displacement gradient in reference coordinates: du/dX = sum_i u_i * dN_i/dX
        du_dX = jnp.dot(dN_dX, u_e)
        F = 1.0 + du_dX  # 1D deformation gradient
        Pi += W_NH_1d(F) * (dx_dxi * w)
    # traction contribution: (optionally) subtract traction * u at right node
    # but we'll handle traction at global level; keep here for completeness
    Pi -= traction_local
    return Pi

In [45]:
element_energy_grad = jax.jit(grad(element_energy, argnums=0))
element_energy_tangent = jax.jit(jacfwd(element_energy_grad, argnums=0))

# -----------------------
# assembly & solver
# -----------------------
def assemble_and_solve(nodes, elems, mu, lam, traction, fixed_dofs, u_fixed_vals,
                       tol=1e-8, maxit=25):
    nnodes = nodes.shape[0]
    # initial guess: zero displacement
    u = jnp.zeros(nnodes)
    # convert to numpy for assembly loops (efficient enough for 1D)
    u = u.copy()
    nodes_np = np.array(nodes)
    elems_np = np.array(elems)
    fixed_dofs = np.array(fixed_dofs, dtype=int)
    # Newton iterations
    for it in range(maxit):
        # assemble global residual R and tangent K
        R = jnp.zeros(nnodes)
        K = jnp.zeros((nnodes, nnodes))
        R = R.at[:].set(0.0)
        K = K.at[:,:].set(0.0)
        # element loop
        for e in range(elems_np.shape[0]):
            nd = elems_np[e]
            X_e = nodes_np[nd]  # reference positions
            u_e = jnp.array([u[nd[0]], u[nd[1]]])
            # compute element residual and tangent
            Re = element_energy_grad(u_e, jnp.array(X_e))
            Ke = element_energy_tangent(u_e, jnp.array(X_e))
            # assemble
            for a in range(2):
                R = R.at[nd[a]].add(Re[a])
                for b in range(2):
                    K = K.at[nd[a], nd[b]].add(Ke[a, b])

        # subtract external traction applied at right-most node: traction * virtual displacement at node -> add negative to R
        # weak form adds -T * delta u at node -> contributes -T to residual at that node
        # we assume traction acts on right end (node index -1)
        R = R.at[-1].add(-traction)

        # apply Dirichlet BCs by modifying residual and stiffness (penalty / elimination)
        # We'll do elimination: zero out rows/cols and set diagonal=1 and residual = u - prescribed
        free_dofs = np.setdiff1d(np.arange(nnodes), fixed_dofs)
        K_np = np.array(K)
        R_np = np.array(R)
        for dof, val in zip(fixed_dofs, u_fixed_vals):
            K_np[dof, :] = 0.0
            K_np[:, dof] = 0.0
            K_np[dof, dof] = 1.0
            R_np[dof] = u[dof] - val  # residual = current - prescribed

        # solve linear system for Newton step
        try:
            delta_u = np.linalg.solve(K_np, -R_np)
        except np.linalg.LinAlgError:
            raise RuntimeError("Singular tangent stiffness matrix during Newton solve.")
        u = u + delta_u

        # convergence check
        norm_R = np.linalg.norm(R_np[free_dofs])
        norm_du = np.linalg.norm(delta_u[free_dofs])
        print(f"Iter {it:2d}: ||R||_free = {norm_R:.3e}, ||du||_free = {norm_du:.3e}")
        if norm_R < tol and norm_du < tol:
            print("Converged.")
            return jnp.array(u)
    raise RuntimeError(f"Newton did not converge after {maxit} iterations. ||R||={norm_R:.3e}")


In [46]:
coords, cells = make_bar(1, 10)

In [67]:
def example_run():
    # bar from X=0 to X=L
    L = 1.0
    n_elems = 2
    nodes, elems = make_bar(L, n_elems)

    # material parameters
    E = 1e3              # Young's modulus (for initial guess mapping to lam, mu)
    nu = 0.3
    mu = E / (2.0*(1.0 + nu))
    lam = E*nu / ((1.0 + nu)*(1.0 - 2.0*nu))  # may be large for near-incompressible

    # boundary conditions
    # left end fixed (u=0)
    fixed_dofs = [0]
    u_fixed_vals = [0.0]
    # traction applied at right end (positive = tensile)
    traction = 2.0

    u = assemble_and_solve(nodes, elems, mu, lam, traction, fixed_dofs, u_fixed_vals,
                           tol=1e-6, maxit=50)
    # print nodal displacements and strains
    print("\nNodal displacement:")
    for i, xi in enumerate(jnp.array(nodes)):
        print(f"  node {i:2d}, X={xi:.4f}, u={float(u[i]):.6f}")

    # compute elemental stretch F average per element
    print("\nElement average F:")
    for e in range(elems.shape[0]):
        nd = elems[e]
        u_e = jnp.array([u[nd[0]], u[nd[1]]])
        X_e = jnp.array(nodes)[nd]
        dx = X_e[1] - X_e[0]
        # central derivative approx: du/dX ~= (u2-u1)/dx
        dudX = (u_e[1] - u_e[0]) / dx
        F = 1.0 + dudX
        print(f" elem {e:2d}, F_avg={float(F):.6f}")

if __name__ == "__main__":
    example_run()

Iter  0: ||R||_free = 1.000e+00, ||du||_free = 2.795e-01
Iter  1: ||R||_free = 0.000e+00, ||du||_free = 0.000e+00
Converged.

Nodal displacement:
  node  0, X=0.0000, u=0.000000
  node  1, X=0.5000, u=0.125000
  node  2, X=1.0000, u=0.250000

Element average F:
 elem  0, F_avg=1.250000
 elem  1, F_avg=1.250000


In [68]:
from core.datasetclass import BenchmarkDataset

/dolfinx-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [84]:
def JacShape(coords_elem) :
    x1, y1 = coords_elem[0]
    x2, y2 = coords_elem[1]
    x3, y3 = coords_elem[2]

    # Jacobian of shape function derivatives
    J = jnp.array([
        [x2 - x1, y2 - y1],
        [x3 - x1, y3 - y1]
    ])
    return J

In [ ]:
def deformation_gradient_element(coords_elem, disp_elem):
    x1, y1 = coords_elem[0]
    x2, y2 = coords_elem[1]
    x3, y3 = coords_elem[2]

    # Jacobian of shape function derivatives
    J = jnp.array([
        [x2 - x1, y2 - y1],
        [x3 - x1, y3 - y1]
    ])

    # Area factor
    detJ = jnp.linalg.det(J)

    # Shape function derivatives in reference space
    dN_ref = jnp.array([
        [-1., -1.],
        [ 1.,  0.],
        [ 0.,  1.]
    ])

    # Convert to physical derivatives: dN/dx = inv(J)^T * dN_ref
    dNdx = jnp.transpose(jnp.linalg.solve(J, dN_ref.T))

    # Gradient of displacement
    gradu = disp_elem.T @ dNdx  # 2x3 @ 3x2 = 2x2

    # Deformation gradient
    F = jnp.eye(2) + gradu
    return F

In [79]:
dataset = BenchmarkDataset("dataset/benchmarks", "noise=low", "NeoHookean")
loadsteps = dataset.loadsteps[:]
F_list = []
invariants_list = []
sigma_list = []
coeffs_list = []
piola_list = []
for i in loadsteps :
    data = dataset[i]
    f = data["F"]
    invariants = data["invariants"]
    sigma = data["sigma"]
    coeffs = data["coeffs"]
    piola = data["P"]
    F_list.append(f)
    invariants_list.append(invariants)
    coeffs_list.append(coeffs)
    sigma_list.append(sigma)
    piola_list.append(piola)

In [83]:
jnp.linalg.det(F_list[0][0][:2,:2])

Array(1.14713655, dtype=float64)

In [81]:
F_list[0][0]

Array([[ 1.04949242, -0.00505564,  0.        ],
       [-0.00611188,  1.09306882,  0.        ],
       [ 0.        ,  0.        ,  1.        ]], dtype=float64)